In [10]:
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))
from wirecoating import WireCoatingDie, CrossLayer


K0 = 273.15
R_W, R_D, L_LAND = 0.577e-3 / 2, 1.25e-3 / 2, 0.10e-3
TAU_C = 2e5                       # critical wall shear stress [Pa]
T_FEP_MIN, T_TOTAL_MAX = 0.20e-3, 0.25e-3


FEP = CrossLayer(eta0=3040, lam=0.0103, m=0.686, E_a=51.4e3, T_ref=330 + K0,
                 name="FEP 9835", rho_melt=1400, rho_solid=2100)
HT = CrossLayer(eta0=12780, lam=1.0345, m=0.529, E_a=92.4e3, T_ref=300 + K0,
                name="HT 2181", rho_melt=1400, rho_solid=1700)


ERR = {("FEP", "eta0"): 40,  ("FEP", "lam"): 0.0007, ("FEP", "m"): 0.028,
       ("HT",  "eta0"): 725, ("HT",  "lam"): 0.2427, ("HT",  "m"): 0.015}

In [11]:

def make_die(V, layers):
    return WireCoatingDie(R_W, R_D, V, list(layers), L=L_LAND)


def max_wall_stress(sol):
    return max(abs(sol.tau_wire), abs(sol.tau_die))


def speed_limit(thicknesses, T, layers):
    """Line speed V* [m/s] where the larger wall shear stress hits TAU_C."""
    state = {"warm": None}

    def g(lnV):
        sol = make_die(np.exp(lnV), layers).solve(thicknesses=thicknesses, T=T,
                                                  warm_start=state["warm"])
        state["warm"] = sol
        return max_wall_stress(sol) / TAU_C - 1.0

    lnV = brentq(g, np.log(1e-4), np.log(20.0), xtol=1e-10)
    return float(np.exp(lnV))


def best_ht(T, fep, ht, t_fep=T_FEP_MIN):
    """Optimal HT thickness [mm] and the speed limit V* [m/s] it gives."""
    upper = T_TOTAL_MAX - t_fep
    res = minimize_scalar(lambda x: -speed_limit([t_fep, x * 1e-3], T, (fep, ht)),
                          bounds=(0.002, upper * 1e3), method="bounded",
                          options={"xatol": 1e-5})
    return res.x, -res.fun



In [12]:
if __name__ == "__main__":
    T = 330 + K0
    base = {"FEP": FEP, "HT": HT}

    t0, V0 = best_ht(T, FEP, HT)
    print(f"Baseline: t_HT* = {t0:.4f} mm, V* = {V0:.4f} m/s\n")

    rows = []
    for (poly, p), pm in ERR.items():
        for sign, end in [(-1, "low"), (+1, "high")]:
            layers = dict(base)
            new_value = getattr(base[poly], p) + sign * pm
            layers[poly] = replace(base[poly], **{p: new_value})
            t_opt, V_opt = best_ht(T, layers["FEP"], layers["HT"])
            rows.append({"polymer": poly, "param": p, "end": end, "value": new_value,
                         "t_HT [mm]": t_opt, "V* [m/s]": V_opt})
            print(f"{poly:>3} {p:<4} {end:<4} -> t_HT* = {t_opt:.4f} mm, V* = {V_opt:.4f} m/s")

    oat = pd.DataFrame(rows)
    out = Path("sensitivity_oat.csv")
    oat.to_csv(out, index=False)
    print(f"\nSaved results to {out}")

Baseline: t_HT* = 0.0285 mm, V* = 0.0568 m/s

FEP eta0 low  -> t_HT* = 0.0285 mm, V* = 0.0569 m/s
FEP eta0 high -> t_HT* = 0.0285 mm, V* = 0.0567 m/s
FEP lam  low  -> t_HT* = 0.0286 mm, V* = 0.0566 m/s
FEP lam  high -> t_HT* = 0.0284 mm, V* = 0.0570 m/s
FEP m    low  -> t_HT* = 0.0286 mm, V* = 0.0568 m/s
FEP m    high -> t_HT* = 0.0284 mm, V* = 0.0568 m/s
 HT eta0 low  -> t_HT* = 0.0285 mm, V* = 0.0632 m/s
 HT eta0 high -> t_HT* = 0.0284 mm, V* = 0.0515 m/s
 HT lam  low  -> t_HT* = 0.0279 mm, V* = 0.0449 m/s
 HT lam  high -> t_HT* = 0.0283 mm, V* = 0.0691 m/s
 HT m    low  -> t_HT* = 0.0286 mm, V* = 0.0493 m/s
 HT m    high -> t_HT* = 0.0280 mm, V* = 0.0665 m/s

Saved results to sensitivity_oat.csv
